# Spotify analysis

In [0]:
dbutils.widgets.text("client_id", "")
dbutils.widgets.text("client_secret", "")
client_id     = dbutils.widgets.get("client_id")
client_secret = dbutils.widgets.get("client_secret")

In [0]:
%restart_python

# Install packages

In [0]:
!pip install spotipy

# Import libraries

In [0]:
# basic packages

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

# spotify packages

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# Invoke Spotify client

In [0]:
# create a Spotify client

client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# Search a playlist

In [0]:
playlist_name = 'regueton'
playlist_results = sp.search(q=playlist_name, type='playlist')

In [0]:
playlist_selected = playlist_results['playlists']['items'][0]

In [0]:
playlist_name_s = playlist_selected['name']
playlist_id_s = playlist_selected['id']
playlist_name_s, playlist_id_s

# Retrieval track data

In [0]:
# playlist_id_s = '5NvuFYJRIJQ9B3bEELa7Yi'

In [0]:
top_tracks = sp.playlist_tracks(playlist_id_s,limit=100)

# Data exploration

In [0]:
top_tracks

In [0]:
top_tracks['items'][0]['track'].keys()

In [0]:
top_tracks['items'][0]['track']['album'].keys()

# Generate playlist

In [0]:
# extract more attributes from each track

list_all_song = []

for track in top_tracks['items']:
    t = track['track']
    if t is None:
        continue

    track_name = t['name']
    track_id = t['id']
    popularity = t['popularity']
    explicit = t['explicit']
    duration_min = t['duration_ms'] / 60000
    track_number = t['track_number']
    num_markets = len(t.get('available_markets', []))

    artists = t['artists']
    artist_name = artists[0]['name']
    artist_id = artists[0]['id']
    all_artists = ', '.join(a['name'] for a in artists)
    num_artists = len(artists)

    album = t['album']
    album_name = album['name']
    album_type = album['album_type']
    total_tracks_album = album['total_tracks']
    release_date = album['release_date']

    list_all_song.append({
        'track_name': track_name,
        'track_id': track_id,
        'artist_name': artist_name,
        'artist_id': artist_id,
        'all_artists': all_artists,
        'num_artists': num_artists,
        'popularity': popularity,
        'explicit': explicit,
        'album_name': album_name,
        'album_type': album_type,
        'total_tracks_album': total_tracks_album,
        'track_number': track_number,
        'num_markets': num_markets,
        'release_date': release_date,
        'duration_min': duration_min,
    })

df_all_songs = pd.DataFrame(list_all_song)
df_all_songs.head()

In [0]:
df_all_songs.shape

In [0]:
df_all_songs.dtypes

In [0]:
# cast date and add release year

df_all_songs['release_date'] = pd.to_datetime(df_all_songs['release_date'], errors='coerce')
df_all_songs['release_year'] = df_all_songs['release_date'].dt.year

In [0]:
def get_bracket_date(x):

    if pd.isna(x):
        return 'Sin fecha'

    if x <= pd.Timestamp('2000-01-01'):
        return 'Antes de los 2000'
    elif x <= pd.Timestamp('2010-01-01'):
        return 'Entre 2000 y 2010'
    elif x <= pd.Timestamp('2020-01-01'):
        return 'Entre 2010 y 2020'
    else:
        return 'Después de 2020' 

In [0]:
df_all_songs['release_date_bracket'] = df_all_songs['release_date'].apply(get_bracket_date)

# Enrich with artist data

Get followers, popularity and genres per artist (one batched call).

In [0]:
# unique artist ids from the playlist

unique_artist_ids = df_all_songs['artist_id'].dropna().unique().tolist()
len(unique_artist_ids)

In [0]:
# batched calls (max 50 per call)

artist_data = {}

for i in range(0, len(unique_artist_ids), 50):
    batch = unique_artist_ids[i:i+50]
    res = sp.artists(batch)
    for a in res['artists']:
        artist_data[a['id']] = {
            'artist_followers': a['followers']['total'],
            'artist_popularity': a['popularity'],
            'artist_genres': ', '.join(a['genres']) if a['genres'] else None,
        }

df_artists = pd.DataFrame.from_dict(artist_data, orient='index').reset_index().rename(columns={'index': 'artist_id'})
df_artists.head()

In [0]:
# merge into main dataframe

df_all_songs = df_all_songs.merge(df_artists, on='artist_id', how='left')
df_all_songs.head()

In [0]:
# df_all_songs.to_csv('regueton_playlist.csv', index=False)

# Descriptive analysis

In [0]:
# the top 10 popular songs

df_all_songs.sort_values('popularity', ascending=False).head(10)

In [0]:
# top 10 most lang songs

df_all_songs.sort_values('duration_min', ascending=False).head(10)

In [0]:
# top 10 most short songs

df_all_songs.sort_values('duration_min', ascending=True).head(10)

In [0]:
# new songs

df_all_songs.sort_values('release_date', ascending=False).head(10)

In [0]:
# old songs

df_all_songs.sort_values('release_date', ascending=True).head(10)

In [0]:
# metrics per artist

df_all_songs.groupby('artist_name').agg({
    'track_name': 'nunique',
    'popularity': 'mean',
    'duration_min': 'mean',
    'release_date': 'max',
    'artist_followers': 'max',
    'artist_popularity': 'max',
}).sort_values('track_name', ascending=False).head(10)

In [0]:
df_all_songs.groupby('release_date_bracket').agg({
    'track_name': 'nunique',
    'popularity': 'mean',
    'duration_min': 'mean',
}).sort_values('popularity', ascending=False)

In [0]:
sns.barplot(x="popularity", y="release_date_bracket", data=df_all_songs)

In [0]:
print(f'popularuty mean: {df_all_songs.popularity.mean():.3f}')

In [0]:
sns.histplot(df_all_songs, x='popularity', kde=True)

In [0]:
# statistics

df_all_songs.groupby('explicit').agg({
    'popularity': ['mean', 'median'],
    'track_name': 'nunique',
    'duration_min': ['mean', 'median'],
})

In [0]:
sns.kdeplot(data=df_all_songs, x="popularity", hue="explicit")

In [0]:
sns.boxplot(data=df_all_songs, x="explicit", y="popularity")

In [0]:
sns.boxplot(data=df_all_songs, x="explicit", y="duration_min")

In [0]:
df_all_songs.head()

In [0]:
sns.scatterplot(data=df_all_songs, x='popularity', y='duration_min', size = 'num_artists')

In [0]:
print('h')

In [0]:
import matplotlib.ticker as ticker

g = sns.lmplot(
    data=df_all_songs,
    x="popularity",
    y="duration_min",
)

In [0]:
import matplotlib.ticker as ticker

g = sns.lmplot(
    data=df_all_songs,
    x="duration_min",
    y="popularity", 
)

In [0]:
sns.regplot(data=df_all_songs, x="duration_min", y="popularity", order=2)

# Collaboration analysis

Does having more artists on a track move popularity?

In [0]:
df_all_songs.groupby('num_artists').agg({
    'track_name': 'nunique',
    'popularity': ['mean', 'median'],
    'duration_min': 'mean',
})

In [0]:
df_all_songs[df_all_songs['num_artists'] > 3]

In [0]:
sns.boxplot(data=df_all_songs, x='num_artists', y='popularity')

In [0]:
# solo vs collab

df_all_songs['is_collab'] = df_all_songs['num_artists'] > 1
df_all_songs.groupby('is_collab').agg({
    'popularity': ['mean', 'median'],
    'track_name': 'nunique',
})

# Album type analysis

Single, album or compilation.

In [0]:
df_all_songs.groupby('album_type').agg({
    'track_name': 'nunique',
    'popularity': ['mean', 'median'],
    'duration_min': 'mean',
    'total_tracks_album': 'mean',
})

In [0]:
sns.boxplot(data=df_all_songs, x='album_type', y='popularity')

In [0]:
sns.countplot(data=df_all_songs, x='album_type')

# Market reach

Number of countries where each track is available.

In [0]:
df_all_songs['num_markets'].describe()

In [0]:
sns.histplot(df_all_songs, x='num_markets', bins=30, kde=True)

In [0]:
sns.scatterplot(data=df_all_songs, x='num_markets', y='popularity', hue='explicit')

# Artist popularity & followers vs track popularity

In [0]:
sns.scatterplot(data=df_all_songs, x='artist_popularity', y='popularity')

In [0]:
# log scale because followers are very skewed

ax = sns.scatterplot(data=df_all_songs, x='artist_followers', y='popularity')
ax.set_xscale('log')

In [0]:
# top 10 artists by followers in this playlist

df_all_songs.groupby('artist_name').agg({
    'artist_followers': 'max',
    'artist_popularity': 'max',
    'popularity': 'mean',
    'track_name': 'nunique',
}).sort_values('artist_followers', ascending=False).head(10)

# Genre analysis

In [0]:
# explode genres (artist_genres is a comma-joined string)

genre_counter = Counter()

for g in df_all_songs['artist_genres'].dropna():
    for genre in g.split(', '):
        genre_counter[genre.strip()] += 1

df_genres = pd.DataFrame(genre_counter.most_common(15), columns=['genre', 'count'])
df_genres

In [0]:
plt.figure(figsize=(8, 6))
sns.barplot(data=df_genres, x='count', y='genre')

# Tracks per year

In [0]:
df_year = df_all_songs.groupby('release_year').agg({
    'track_name': 'nunique',
    'popularity': 'mean',
}).reset_index()

df_year

In [0]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_year, x='release_year', y='track_name', marker='o')

In [0]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_year, x='release_year', y='popularity', marker='o')

# Correlation heatmap

Numeric attributes against each other.

In [0]:
num_cols = [
    'popularity',
    'duration_min',
    'num_artists',
    'num_markets',
    'total_tracks_album',
    'track_number',
    'artist_followers',
    'artist_popularity',
    'release_year',
]

plt.figure(figsize=(9, 7))
sns.heatmap(df_all_songs[num_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')